# Fig S5E — TF motif accessibility across pseudotime (full heatmap)
Full heatmap companion to main-figure **Fig 5D**; based on **08_PlotTFActivity Part C**. Per pseudotime segment (vs the NE root `PT_seg01`), OLS coef of peak Log2FC on the grouped motif matrix. Rows = ALL TFs significant (FDR≤0.1) in **≥1 segment**, ordered by Spearman trend across segments — **rising at top, falling at bottom**; columns run root NE → Differentiated tip. `*` = FDR≤0.1.

In [ ]:
from paperfig_style import *
import numpy as np, pandas as pd, matplotlib.pyplot as plt


In [ ]:
from scipy.stats import spearmanr
# --- 08_PlotTFActivity Part C (cells 10-11), replicated ----------------------
pcoef = load_matrix('PseudotimeTFActivity_coef.csv')
pfdr  = load_matrix('PseudotimeTFActivity_FDR.csv')
seg = sorted(pcoef.columns, key=lambda c: int(c.replace('PT_seg', '')))
pcoef, pfdr = pcoef[seg], pfdr[seg]
MIN_SIG = 1   # supplementary full heatmap: keep TFs significant in >=1 segment
sigTF = pcoef.index[(pfdr <= FDR_THRESHOLD).sum(axis=1) >= MIN_SIG]
pmat = pcoef.loc[sigTF]; pstar = (pfdr.loc[sigTF] <= FDR_THRESHOLD).values
idx = np.arange(pmat.shape[1])
trend = np.array([spearmanr(idx, pmat.values[i])[0] for i in range(len(pmat))])
order = np.argsort(-trend)                                         # rising -> falling
pmat = pmat.iloc[order]; pstar = pstar[order]
print(len(sigTF), 'sig TFs across', pmat.shape[1], 'pseudotime segments (root NE -> Diff tip)')
lim = np.abs(pmat.values).max()
fig, ax = plt.subplots(figsize=(3.8, max(4.0, 0.14*len(pmat))), layout='constrained')
im = ax.imshow(pmat.values, aspect='auto', cmap=ACTIVITY_CMAP, vmin=-lim, vmax=lim)
ax.set_xticks(range(0, pmat.shape[1], 2)); ax.set_xticklabels(range(1, pmat.shape[1]+1, 2))
ax.set_xlabel('pseudotime segment (root NE \u2192 Differentiated tip)')
ax.set_yticks(range(len(pmat))); ax.set_yticklabels(tf_labels(pmat.index))
for i in range(pmat.shape[0]):
    for j in range(pmat.shape[1]):
        if pstar[i, j]: ax.text(j, i, '*', ha='center', va='center', fontsize=7, color='black')
ax.tick_params(length=0); [s.set_visible(False) for s in ax.spines.values()]
# horizontal colorbar at the bottom (fits the tall heatmap better than a side bar)
cb = fig.colorbar(im, ax=ax, location='bottom', fraction=0.03, pad=0.06, aspect=35)
cb.set_label('TF motif accessibility (coef)'); cb.outline.set_linewidth(0.5)
ax.set_title('TF motif accessibility across pseudotime\n(root NE \u2192 Differentiated tip;  * FDR\u22640.1)', fontsize=8)
savepanel(fig, 'FigS5E_PseudotimeTFActivity_heatmap')
